In [1]:
import sqlite3
import logging
import os
import pandas as pd
from datetime import datetime

# Set up logging
logging.basicConfig(filename='database_operations.log', level=logging.INFO, 
                    format='%(asctime)s - %(levelname)s - %(message)s')

def connect_to_db(db_path):
    try:
        if not os.path.exists(db_path):
            logging.info(f"Creating new database: {db_path}")
        conn = sqlite3.connect(db_path)
        logging.info(f"Successfully connected to {db_path}")
        return conn
    except sqlite3.Error as e:
        logging.error(f"Error connecting to {db_path}: {e}")
        raise

# Connect to databases
try:
    sdm_conn = connect_to_db('GreatOutdoors_SDM.db')
    dwh_conn = connect_to_db('GreatOutdoors_DWH.db')
    # Note: Keep connections open for use in subsequent cells
except Exception as e:
    logging.critical(f"Failed to establish database connections: {e}")
    # Handle or re-raise as needed

In [2]:
cur = dwh_conn.cursor()
cur_sdm = sdm_conn.cursor()

In [4]:
dwh_conn.executescript("""
            -- Dimension / lookup tables first (no FK dependencies)
CREATE TABLE IF NOT EXISTS "DATE" (
    DATE_KEY  INTEGER PRIMARY KEY AUTOINCREMENT,
    YEAR      INT NOT NULL,
    MONTH     INT NOT NULL,
    DAY       INT NOT NULL
);

CREATE TABLE IF NOT EXISTS CUSTOMER (
    CUSTOMER_SK                 INTEGER PRIMARY KEY AUTOINCREMENT,
    CUSTOMER_CODE               VARCHAR(50)     NOT NULL,
    COUNTRY                     VARCHAR(100),
    CITY                        VARCHAR(100),
    REGION                      VARCHAR(100),
    POSTAL_ZONE                 VARCHAR(20),
    ACTIVE_INDICATOR            CHAR(1),
    COMPANY_NAME                VARCHAR(200),
    SEGMENT_NAME                VARCHAR(100),
    CUSTOMER_TYPE_EN            VARCHAR(100),
    CUSTOMER_HEADQUARTERS_CODE  VARCHAR(50),
    TERRITORY                   VARCHAR(100),
    CUSTOMER_SITE_CODE          VARCHAR(50),
    EFFECTIVE_FROM              DATE,
    EFFECTIVE_TO                DATE,
    ACTIVE                      CHAR(1)
);

CREATE TABLE IF NOT EXISTS PRODUCT (
    PRODUCT_SK              INTEGER PRIMARY KEY AUTOINCREMENT,
    PRODUCT_NUMBER          VARCHAR(50)     NOT NULL,
    INTRODUCTION_DATE       DATE,
    AGE                     INT,
    PRODUCTION_COST_GROUP   VARCHAR(50),
    MARGIN_GROUP            VARCHAR(50),
    PRODUCT_NAME            VARCHAR(200),
    PRODUCT_LINE_EN         VARCHAR(100),
    PRODUCT_TYPE_EN         VARCHAR(100),
    EFFECTIVE_FROM          DATE,
    EFFECTIVE_TO            DATE,
    ACTIVE                  CHAR(1)
);

CREATE TABLE IF NOT EXISTS CUSTOMER_AGE_GROUP (
    AGE_GROUP_CODE  VARCHAR(50)     NOT NULL,
    UPPER_AGE       INT,
    LOWER_AGE       INT,
    CONSTRAINT PK_CUSTOMER_AGE_GROUP PRIMARY KEY (AGE_GROUP_CODE)
);

CREATE TABLE IF NOT EXISTS ORDER_DETAILS (
    ORDER_DETAIL_CODE   VARCHAR(50)     NOT NULL,
    ORDER_NUMBER        VARCHAR(50),
    DATE_KEY            INT,
    ORDER_METHOD        VARCHAR(50),
    CUSTOMER_CODE       VARCHAR(50),
    PRODUCT_NUMBER      VARCHAR(50),
    QUANTITY            INT,
    UNIT_COST           DECIMAL(18,2),
    UNIT_PRICE          DECIMAL(18,2),
    UNIT_SALE_PRICE     DECIMAL(18,2),
    OMZET               DECIMAL(18,2),
    WINST               DECIMAL(18,2),
    CONTRIBUTION_MARGIN DECIMAL(18,2),
    CONSTRAINT PK_ORDER_DETAILS PRIMARY KEY (ORDER_DETAIL_CODE),
    CONSTRAINT FK_OD_DATE     FOREIGN KEY (DATE_KEY)       REFERENCES "DATE" (DATE_KEY),
    CONSTRAINT FK_OD_CUSTOMER FOREIGN KEY (CUSTOMER_CODE)  REFERENCES CUSTOMER (CUSTOMER_CODE),
    CONSTRAINT FK_OD_PRODUCT  FOREIGN KEY (PRODUCT_NUMBER) REFERENCES PRODUCT (PRODUCT_NUMBER)
);

CREATE TABLE IF NOT EXISTS RETURNED_ITEM (
    RETURN_CODE         VARCHAR(50)     NOT NULL,
    DATE_KEY            INT,
    RETURN_REASON       VARCHAR(200),
    ORDER_DETAIL_CODE   VARCHAR(50),
    RETURN_QUANTITY     INT,
    CONSTRAINT PK_RETURNED_ITEM PRIMARY KEY (RETURN_CODE),
    CONSTRAINT FK_RI_ORDER_DETAIL FOREIGN KEY (ORDER_DETAIL_CODE) REFERENCES ORDER_DETAILS (ORDER_DETAIL_CODE),
    CONSTRAINT FK_RI_DATE FOREIGN KEY (DATE_KEY) REFERENCES "DATE" (DATE_KEY)
);

CREATE TABLE IF NOT EXISTS SALES_DEMOGRAPHIC (
    DEMOGRAPHIC_CODE    VARCHAR(50)     NOT NULL,
    CUSTOMER_CODE       VARCHAR(50),
    AGE_GROUP_CODE      VARCHAR(50),
    SALES_PERCENT       DECIMAL(10,4),
    CONSTRAINT PK_SALES_DEMOGRAPHIC PRIMARY KEY (DEMOGRAPHIC_CODE),
    CONSTRAINT FK_SD_CUSTOMER  FOREIGN KEY (CUSTOMER_CODE)  REFERENCES CUSTOMER (CUSTOMER_CODE),
    CONSTRAINT FK_SD_AGE_GROUP FOREIGN KEY (AGE_GROUP_CODE) REFERENCES CUSTOMER_AGE_GROUP (AGE_GROUP_CODE)
);

CREATE TABLE IF NOT EXISTS INVENTORY (
    DATE_KEY        INT         NOT NULL,
    PRODUCT_KEY     VARCHAR(50) NOT NULL,
    INVENTORY_COUNT INT,
    CONSTRAINT PK_INVENTORY PRIMARY KEY (DATE_KEY, PRODUCT_KEY),
    CONSTRAINT FK_INV_DATE    FOREIGN KEY (DATE_KEY)    REFERENCES "DATE" (DATE_KEY),
    CONSTRAINT FK_INV_PRODUCT FOREIGN KEY (PRODUCT_KEY) REFERENCES PRODUCT (PRODUCT_NUMBER)
);

CREATE TABLE IF NOT EXISTS PRODUCT_FORECAST (
    PRODUCT_KEY     VARCHAR(50) NOT NULL,
    DATE_KEY        INT         NOT NULL,
    EXPECTED_VOLUME DECIMAL(18,2),
    CONSTRAINT PK_PRODUCT_FORECAST PRIMARY KEY (PRODUCT_KEY, DATE_KEY),
    CONSTRAINT FK_PF_PRODUCT FOREIGN KEY (PRODUCT_KEY) REFERENCES PRODUCT (PRODUCT_NUMBER),
    CONSTRAINT FK_PF_DATE    FOREIGN KEY (DATE_KEY)    REFERENCES "DATE" (DATE_KEY)
);
""")

In [3]:
#gewoon voor testen

dwh_conn.executescript("""
DROP TABLE IF EXISTS PRODUCT_FORECAST;
DROP TABLE IF EXISTS INVENTORY;
DROP TABLE IF EXISTS SALES_DEMOGRAPHIC;
DROP TABLE IF EXISTS RETURNED_ITEM;
DROP TABLE IF EXISTS ORDER_DETAILS;
DROP TABLE IF EXISTS CUSTOMER_AGE_GROUP;
DROP TABLE IF EXISTS CUSTOMER;
DROP TABLE IF EXISTS PRODUCT;
DROP TABLE IF EXISTS DATE;
                       """)

In [5]:
# ETL datums
def etl_datums():
    logging.info("START ETL: DATE")

    dates = pd.read_sql("""
        SELECT INVENTORY_YEAR AS YEAR, INVENTORY_MONTH AS MONTH FROM INVENTORY_LEVELS
        UNION
        SELECT YEAR, MONTH FROM PRODUCT_FORECAST
    """, sdm_conn)
    logging.info(f"  Csv datums geladen: {len(dates)} rijen")

    dates["DAY"] = 1

    sale_dates = pd.read_sql("""
        SELECT RETURN_DATE AS DATE FROM SALES_returned_item
        UNION
        SELECT ORDER_DATE AS DATE FROM SALES_order_header
    """, sdm_conn)
    logging.info(f"  Inkoop datums geladen: {len(sale_dates)} rijen")

    dt = pd.to_datetime(sale_dates["DATE"], format="%d-%b-%Y %I:%M:%S %p")
    sale_dates["YEAR"]  = dt.dt.year
    sale_dates["MONTH"] = dt.dt.month
    sale_dates["DAY"]   = dt.dt.day
    sale_dates = sale_dates.drop(columns=["DATE"])

    all_dates = pd.concat([dates, sale_dates], ignore_index=True)
    all_dates = all_dates.drop_duplicates()
    logging.info(f"  Unieke datums na deduplicatie: {len(all_dates)}")

    inserted = 0
    for _, rij in all_dates.iterrows():
        cur.execute("""
            INSERT OR IGNORE INTO "DATE" (YEAR, MONTH, DAY)
            VALUES (?, ?, ?)
        """, [int(rij["YEAR"]), int(rij["MONTH"]), int(rij["DAY"])])
        inserted += cur.rowcount

    dwh_conn.commit()
    logging.info(f"  Ingevoegd in DATE: {inserted} rijen")
    logging.info("KLAAR ETL: DATE")

In [6]:
def etl_products():
    cur_sdm.execute("""
        SELECT
            p.PRODUCT_NUMBER,
            p.INTRODUCTION_DATE,
            p.PRODUCTION_COST,
            p.MARGIN,
            p.PRODUCT_NAME,
            pt.PRODUCT_LINE_CODE,
            pl.PRODUCT_LINE_EN,
            pt.PRODUCT_TYPE_EN
        FROM SALES_product p
        JOIN SALES_product_type pt ON p.PRODUCT_TYPE_CODE = pt.PRODUCT_TYPE_CODE
        JOIN SALES_product_line pl ON pt.PRODUCT_LINE_CODE = pl.PRODUCT_LINE_CODE
        WHERE p.LANGUAGE = 'EN'
    """)

    today = datetime.today().strftime('%Y-%m-%d')
    inserted = updated = skipped = 0

    for row in cur_sdm.fetchall():
        product_number, intro_date, prod_cost, margin, name, \
        line_code, line_en, type_en = row

        try:
            intro = datetime.strptime(intro_date, '%d-%b-%Y %I:%M:%S %p')
            age = datetime.today().year - intro.year
        except (ValueError, TypeError):
            age = None

        cost = float(prod_cost) if prod_cost else 0
        if cost < 10:
            cost_group = 'Low'
        elif cost < 50:
            cost_group = 'Medium'
        elif cost < 200:
            cost_group = 'High'
        else:
            cost_group = 'Very High'

        mgn = float(margin) if margin else 0
        if mgn < 0.25:
            margin_group = 'Low'
        elif mgn < 0.40:
            margin_group = 'Medium'
        else:
            margin_group = 'High'

        # Check for existing active record
        cur.execute("""
            SELECT PRODUCT_SK, PRODUCT_NAME, PRODUCT_LINE_EN, PRODUCT_TYPE_EN,
                   PRODUCTION_COST_GROUP, MARGIN_GROUP
            FROM PRODUCT
            WHERE PRODUCT_NUMBER = ? AND ACTIVE = 1
        """, (product_number,))
        existing = cur.fetchone()

        if existing is None:
            # New product — insert
            cur.execute("""
                INSERT INTO PRODUCT (
                    PRODUCT_NUMBER, INTRODUCTION_DATE, AGE,
                    PRODUCTION_COST_GROUP, MARGIN_GROUP,
                    PRODUCT_NAME, PRODUCT_LINE_EN, PRODUCT_TYPE_EN,
                    EFFECTIVE_FROM, EFFECTIVE_TO, ACTIVE
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, NULL, 1)
            """, (product_number, intro_date, age, cost_group, margin_group,
                  name, line_en, type_en, today))
            inserted += 1

        else:
            sk, ex_name, ex_line, ex_type, ex_cost_grp, ex_margin_grp = existing

            # Define which fields trigger a new SCD2 version
            changed = (
                ex_name      != name        or
                ex_line      != line_en     or
                ex_type      != type_en     or
                ex_cost_grp  != cost_group  or
                ex_margin_grp != margin_group
            )

            if changed:
                # Expire old record
                cur.execute("""
                    UPDATE PRODUCT SET EFFECTIVE_TO = ?, ACTIVE = 0
                    WHERE PRODUCT_SK = ?
                """, (today, sk))

                # Insert new version
                cur.execute("""
                    INSERT INTO PRODUCT (
                        PRODUCT_NUMBER, INTRODUCTION_DATE, AGE,
                        PRODUCTION_COST_GROUP, MARGIN_GROUP,
                        PRODUCT_NAME, PRODUCT_LINE_EN, PRODUCT_TYPE_EN,
                        EFFECTIVE_FROM, EFFECTIVE_TO, ACTIVE
                    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, NULL, 1)
                """, (product_number, intro_date, age, cost_group, margin_group,
                      name, line_en, type_en, today))
                updated += 1
            else:
                skipped += 1

    dwh_conn.commit()
    logging.info(f"PRODUCT: {inserted} inserted, {updated} updated (SCD2), {skipped} unchanged")
    print(f"PRODUCT: {inserted} inserted, {updated} updated (SCD2), {skipped} unchanged")

In [ ]:
def etl_order_details():
    cur_sdm.execute("""
    SELECT
        od.ORDER_DETAIL_CODE,
        od.ORDER_NUMBER,
        oh.ORDER_DATE,
        oh.ORDER_METHOD_CODE,
        om.ORDER_METHOD_EN,
        rs.RETAILER_SITE_CODE,
        od.PRODUCT_NUMBER,
        od.QUANTITY,
        od.UNIT_COST,
        od.UNIT_PRICE,
        od.UNIT_SALE_PRICE
    FROM SALES_order_details od
    JOIN SALES_order_header oh ON od.ORDER_NUMBER = oh.ORDER_NUMBER
    JOIN SALES_order_method om ON oh.ORDER_METHOD_CODE = om.ORDER_METHOD_CODE
    JOIN SALES_retailer_site rs ON oh.RETAILER_SITE_CODE = rs.RETAILER_SITE_CODE
    """)

    rows = []
    skipped = 0

    for row in cur_sdm.fetchall():
        detail_code, order_number, order_date, method_code, method_en, \
        customer_code, product_number, qty, unit_cost, unit_price, unit_sale_price = row

        # Parse date and look up DATE_KEY
        try:
            d = datetime.strptime(order_date, '%d-%b-%Y %I:%M:%S %p')
            cur.execute(
                "SELECT DATE_KEY FROM DATE WHERE YEAR=? AND MONTH=? AND DAY=?",
                (d.year, d.month, d.day)
            )
            result = cur.fetchone()
            date_key = result[0] if result else None
        except (ValueError, TypeError):
            date_key = None

        # Cast to float for calculations
        qty_f        = float(qty)        if qty        else 0
        cost_f       = float(unit_cost)  if unit_cost  else 0
        sale_price_f = float(unit_sale_price) if unit_sale_price else 0

        omzet               = round(qty_f * sale_price_f, 2)
        winst               = round(qty_f * (sale_price_f - cost_f), 2)
        contribution_margin = round(winst / omzet, 4) if omzet != 0 else None

        rows.append((
            detail_code,
            order_number,
            date_key,
            method_en,
            str(customer_code),
            product_number,
            qty_f,
            cost_f,
            float(unit_price) if unit_price else None,
            sale_price_f,
            omzet,
            winst,
            contribution_margin
        ))

    cur.executemany("""
        INSERT OR IGNORE INTO ORDER_DETAILS (
            ORDER_DETAIL_CODE, ORDER_NUMBER, DATE_KEY,
            ORDER_METHOD, CUSTOMER_CODE, PRODUCT_NUMBER,
            QUANTITY, UNIT_COST, UNIT_PRICE, UNIT_SALE_PRICE,
            OMZET, WINST, CONTRIBUTION_MARGIN
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, rows)

    dwh_conn.commit()
    logging.info(f"ORDER_DETAILS: inserted {cur.rowcount} rows")
    print(f"ORDER_DETAILS: {len(rows)} rows processed, {skipped} skipped")

In [ ]:
def etl_returned_items():
    cur_sdm.execute("""
    SELECT
        ri.RETURN_CODE,
        ri.RETURN_DATE,
        rr.RETURN_DESCRIPTION_EN,
        ri.ORDER_DETAIL_CODE,
        ri.RETURN_QUANTITY
    FROM SALES_returned_item ri
    JOIN SALES_return_reason rr ON ri.RETURN_REASON_CODE = rr.RETURN_REASON_CODE
    """)

    rows = []

    for row in cur_sdm.fetchall():
        return_code, return_date_raw, return_reason, order_detail_code, return_qty = row

        # Normalise date to ISO format
        try:
            d = datetime.strptime(return_date_raw, '%d-%b-%Y %I:%M:%S %p')
            return_date = d.strftime('%Y-%m-%d')
        except (ValueError, TypeError):
            return_date = None

        rows.append((
            return_code,
            return_date,
            return_reason,
            order_detail_code,
            int(return_qty) if return_qty else None
        ))

    cur.executemany("""
        INSERT OR IGNORE INTO RETURNED_ITEM (
            RETURN_CODE, RETURN_DATE, RETURN_REASON,
            ORDER_DETAIL_CODE, RETURN_QUANTITY
        ) VALUES (?, ?, ?, ?, ?)
    """, rows)

    dwh_conn.commit()
    logging.info(f"RETURNED_ITEM: inserted {cur.rowcount} rows")
    print(f"RETURNED_ITEM: {len(rows)} rows processed")

In [7]:
etl_datums()
etl_products()

PRODUCT: 115 inserted, 0 updated (SCD2), 0 unchanged
